# LSTM experiments: which features, and does augmentation help?

This notebook is the **second stage of the model comparison**. A recurrent network (LSTM) is trained on the same acoustic features as the classical models, to answer three questions:

1. Which features work best with an LSTM (MFCC, chroma, mel, or a combination)?
2. Does training on the **augmented** set (5,587 clips, with extra and augmented `angry`, `happy` and `sad` clips) beat the **original imbalanced** set (3,881 clips)?
3. Does a deeper network help?

> **About the outputs.** The training logs and numbers stored here come from the original runs on Google Colab (2022). The cleaned code was not re-run on the original audio (the corpus is not part of this repository), so **cells without a stored output were added during the clean-up and are untested on the original data**. Two changes were made to the code: `tensorflow_addons` (discontinued) was replaced by the built-in Keras `F1Score` metric, and layer definitions use `keras.Input`.

> **A note on the design.** Each clip is first summarised as a vector (the average of each feature over time), and the LSTM then reads this vector as a sequence of length 40 (MFCC), 12 (chroma), 128 (mel) and so on. The recurrence therefore runs over the **feature dimension, not over time**. Feeding frame-level sequences instead is the natural next step (see the README).

## 1. Setup

In [ ]:
from pathlib import Path

DATA_DIR = Path("data")                       # same layout as notebook 02 (train/, val/, annotations.xlsx)
ANNOTATION_FILE = DATA_DIR / "annotations.xlsx"
TRAIN_GLOB, TEST_GLOB = "train/**/*.wav", "val/**/*.wav"

FEATURES = dict(mfcc=True, chroma=False, mel=False)   # edit to compare feature sets: 40 / 12 / 128 values
EMOTIONS = ["angry", "happy", "neutral", "sad"]
EPOCHS, BATCH_SIZE = 200, 512

In [ ]:
import numpy as np
import pandas as pd
import librosa
import soundfile
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow import keras
from tensorflow.keras import layers

## 2. Data

Same feature extraction and loading code as notebook 02 (`extract_feature`, `load_data`), with the labels one-hot encoded for the network. The test clips are also used as the validation data during training, as in the original runs.

In [ ]:
def extract_feature(file_name, mfcc=False, chroma=False, mel=False):
    """Return one vector for a wav file: the time-average of the selected features."""
    with soundfile.SoundFile(str(file_name)) as sound_file:
        X = sound_file.read(dtype="float32")
        sample_rate = sound_file.samplerate
    if X.ndim > 1:
        X = X.mean(axis=1)
    result = np.array([])
    if mfcc:
        result = np.hstack((result, np.mean(librosa.feature.mfcc(y=X, sr=sample_rate, n_mfcc=40).T, axis=0)))
    if chroma:
        stft = np.abs(librosa.stft(X))
        result = np.hstack((result, np.mean(librosa.feature.chroma_stft(S=stft, sr=sample_rate).T, axis=0)))
    if mel:
        result = np.hstack((result, np.mean(librosa.feature.melspectrogram(y=X, sr=sample_rate).T, axis=0)))
    return result


annotations = pd.read_excel(ANNOTATION_FILE)
label_of = dict(zip(annotations["path"], annotations["label"]))


def load_data(pattern):
    X, y, missing = [], [], 0
    for file in sorted(DATA_DIR.glob(pattern)):
        label = label_of.get(file.name)
        if label is None:
            missing += 1
            continue
        if label not in EMOTIONS:
            continue
        X.append(extract_feature(file, **FEATURES))
        y.append(EMOTIONS.index(label))
    if missing:
        print(f"warning: {missing} file(s) without annotation were skipped")
    return np.array(X), np.array(y)

In [ ]:
X_tr, y_tr = load_data(TRAIN_GLOB)
X_ts, y_ts = load_data(TEST_GLOB)

# the LSTM expects (samples, timesteps, channels); labels are one-hot vectors
X_train, X_test = X_tr[..., np.newaxis], X_ts[..., np.newaxis]
y_train = keras.utils.to_categorical(y_tr, num_classes=len(EMOTIONS))
y_test = keras.utils.to_categorical(y_ts, num_classes=len(EMOTIONS))
print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)

(5587, 40, 1) (5587, 4) (1025, 40, 1) (1025, 4)


## 3. Baseline LSTM

One LSTM layer (123 units) followed by two dense layers with dropout. Training uses Adam, categorical cross-entropy, a batch size of 512 and **200 epochs**. Accuracy and **macro-F1** are tracked at each epoch, on the training and on the test clips.

In [ ]:
def build_lstm(n_features, dropout=0.3):
    model = keras.Sequential([
        keras.Input(shape=(n_features, 1)),
        layers.LSTM(123),
        layers.Dense(64, activation="relu"),
        layers.Dropout(dropout),
        layers.Dense(32, activation="relu"),
        layers.Dropout(dropout),
        layers.Dense(len(EMOTIONS), activation="softmax"),
    ])
    model.compile(
        loss="categorical_crossentropy",
        optimizer="adam",
        metrics=["accuracy", keras.metrics.F1Score(average="macro", name="f1_score")],
    )
    return model


model = build_lstm(X_train.shape[1])
model.summary()

In [ ]:
history = model.fit(X_train, y_train, validation_data=(X_test, y_test),
                    epochs=EPOCHS, batch_size=BATCH_SIZE, shuffle=True)

Epoch 1/200
11/11 [==============================] - 19s 642ms/step - loss: 1.3783 - accuracy: 0.2744 - f1_score: 0.2662 - val_loss: 1.3396 - val_accuracy: 0.3698 - val_f1_score: 0.2567
Epoch 2/200
11/11 [==============================] - 3s 272ms/step - loss: 1.3537 - accuracy: 0.3424 - f1_score: 0.3233 - val_loss: 1.3515 - val_accuracy: 0.3463 - val_f1_score: 0.2601
... (epochs 3 to 197 omitted) ...
Epoch 198/200
11/11 [==============================] - 3s 280ms/step - loss: 0.1063 - accuracy: 0.9640 - f1_score: 0.9640 - val_loss: 3.1575 - val_accuracy: 0.5424 - val_f1_score: 0.4910
Epoch 199/200
11/11 [==============================] - 3s 288ms/step - loss: 0.1210 - accuracy: 0.9554 - f1_score: 0.9554 - val_loss: 3.1696 - val_accuracy: 0.5502 - val_f1_score: 0.4963
Epoch 200/200
11/11 [==============================] - 3s 291ms/step - loss: 0.1366 - accuracy: 0.9506 - f1_score: 0.9506 - val_loss: 3.1505 - val_accuracy: 0.5444 - val_f1_score: 0.4935


*Stored output above: the original run with MFCC features and the augmented training set (epochs 3 to 197 omitted here). The training F1 reaches 0.95 while the test F1 stays around 0.49: the network memorises the training clips.*

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
for ax, (key, title) in zip(axes, [("loss", "Loss"), ("accuracy", "Accuracy"), ("f1_score", "Macro-F1")]):
    ax.plot(history.history[key], label="training")
    ax.plot(history.history["val_" + key], label="test")
    ax.set_title(title)
    ax.set_xlabel("epoch")
    ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
predictions = np.argmax(model.predict(X_test), axis=1)
print(classification_report(y_ts, predictions, target_names=EMOTIONS))
print(confusion_matrix(y_ts, predictions))

## 4. Experiments: features and augmentation

The notebook above was run once per feature set, on both the **imbalanced** training set (3,881 original clips) and the **augmented** one (5,587 clips: the original clips plus extra and augmented `angry`, `happy` and `sad` clips). All runs: 200 epochs, same LSTM(123) architecture (dropout 0.2 or 0.3 depending on the run), same 1,025 test clips. The values below are those of the **last epoch**, read from the stored logs.

| Features | Values per clip | Training set (clips) | Training F1 | Test accuracy | Test macro-F1 |
|---|---|---|---|---|---|
| MFCC | 40 | imbalanced (3,881) | 0.96 | 0.54 | 0.49 |
| MFCC | 40 | augmented (5,587) | 0.95 | 0.54 | 0.49 |
| Chroma | 12 | imbalanced (3,881) | 0.31 | 0.43 | 0.29 |
| Chroma | 12 | augmented (5,587) | 0.38 | 0.36 | 0.34 |
| Mel | 128 | imbalanced (3,881) | 0.46 | 0.51 | 0.40 |
| Mel | 128 | augmented (5,587) | 0.85 | 0.48 | 0.44 |
| MFCC+Chroma | 52 | imbalanced (3,881) | 0.87 | 0.53 | 0.49 |
| MFCC+Chroma | 52 | augmented (5,587) | 0.85 | 0.53 | 0.47 |
| MFCC+Chroma+Mel | 180 | imbalanced (3,881) | 0.58 | 0.50 | 0.41 |
| MFCC+Chroma+Mel | 180 | augmented (5,587) | 0.75 | 0.52 | 0.46 |

![LSTM feature comparison](../results/lstm_feature_comparison.png)

**What to take from it**
- **MFCC is the strongest feature set in these logs** (test macro-F1 0.49). Adding chroma or mel does not help, and chroma alone is clearly the weakest (0.29 to 0.34). The thesis tables come from other runs and score mel (accuracy 0.61, F1 0.60) and MFCC+chroma (0.62, 0.58) as high as MFCC (0.62, 0.56), so the ranking of those three is not reliable; chroma alone is the weakest in both.
- **Augmentation helps the weaker feature sets** (chroma +0.04, mel +0.04, MFCC+chroma+mel +0.06) but **makes no real difference for MFCC** (0.486 to 0.493) and slightly hurts MFCC+chroma (0.487 to 0.469).
- **All models overfit heavily**: the training F1 is far above the test F1 (0.95 vs 0.49 for MFCC).

![Training vs test F1](../results/lstm_overfitting.png)

## 5. Deeper architecture

A deeper network was tried on MFCC features and the augmented training set: three stacked LSTM layers (100, 200 and 100 units) with batch normalisation, recurrent dropout, average pooling and the RMSprop optimiser, trained for 100 epochs with a batch size of 32. (The diagram of this network in the thesis has the same three layers but no pooling layer: the flattened output has 40 x 100 = 4,000 values. The log kept below is the run with average pooling.)

In [ ]:
def build_stacked_lstm(n_features, n_dim=100):
    model = keras.Sequential([
        keras.Input(shape=(n_features, 1)),
        layers.BatchNormalization(),
        layers.LSTM(n_dim, return_sequences=True, dropout=0.1, recurrent_dropout=0.2),
        layers.LSTM(n_dim * 2, return_sequences=True, dropout=0.1, recurrent_dropout=0.2),
        layers.LSTM(n_dim, return_sequences=True, dropout=0.1, recurrent_dropout=0.2),
        layers.AveragePooling1D(pool_size=2),
        layers.Flatten(),
        layers.Dense(len(EMOTIONS), activation="softmax"),
    ])
    model.compile(
        loss="categorical_crossentropy",
        optimizer="rmsprop",
        metrics=["accuracy", keras.metrics.F1Score(average="macro", name="f1_score")],
    )
    return model


stacked = build_stacked_lstm(X_train.shape[1])
history = stacked.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=100, batch_size=32)
stacked.evaluate(X_test, y_test, batch_size=32)

Epoch 1/100
175/175 [==============================] - 98s 492ms/step - loss: 1.2755 - accuracy: 0.4133 - f1_score: 0.3892 - val_loss: 1.2994 - val_accuracy: 0.4205 - val_f1_score: 0.3550
Epoch 2/100
175/175 [==============================] - 89s 509ms/step - loss: 1.2299 - accuracy: 0.4407 - f1_score: 0.4211 - val_loss: 1.2841 - val_accuracy: 0.3990 - val_f1_score: 0.3497
... (epochs 3 to 97 omitted) ...
Epoch 98/100
175/175 [==============================] - 89s 508ms/step - loss: 0.3886 - accuracy: 0.8461 - f1_score: 0.8469 - val_loss: 1.1658 - val_accuracy: 0.6195 - val_f1_score: 0.5851
Epoch 99/100
175/175 [==============================] - 92s 525ms/step - loss: 0.3844 - accuracy: 0.8452 - f1_score: 0.8462 - val_loss: 1.1241 - val_accuracy: 0.6224 - val_f1_score: 0.5737
Epoch 100/100
175/175 [==============================] - 93s 532ms/step - loss: 0.3806 - accuracy: 0.8531 - f1_score: 0.8539 - val_loss: 1.1714 - val_accuracy: 0.6302 - val_f1_score: 0.5751
0.8530517220497131 0.63

*Stored output above: the original run (epochs 3 to 97 omitted). The last line is `[loss, accuracy, macro-F1]` on the test clips.*

| Variant (MFCC, augmented set) | Epochs | Training F1 | Test accuracy | Test macro-F1 |
|---|---|---|---|---|
| **3 stacked LSTM + batch norm, RMSprop** | 100 | 0.85 | **0.63** | **0.58** |
| 2 stacked LSTM, no dense head | 55 | 0.77 | 0.57 | 0.51 |
| LSTM(123) + dense head | 65 | 0.73 | 0.59 | 0.52 |
| LSTM(123) + dense head | 55 | 0.64 | 0.58 | 0.50 |
| LSTM(123) + dense head (baseline above) | 200 | 0.95 | 0.54 | 0.49 |

The deeper network is the best of all the LSTMs (test accuracy 0.63, macro-F1 0.58), and shorter training (55 to 100 epochs) beats 200 epochs. It remains below fine-tuned wav2vec 2.0 (accuracy 0.69, macro-F1 0.68, notebook 04).

## 6. Bidirectional LSTM

The thesis also compares a **bidirectional LSTM** (BiLSTM). Its code did not survive, but the thesis gives the architecture (a diagram with the layer shapes): the 40 MFCC values, one bidirectional LSTM layer whose two directions of 123 units give 246 outputs, a dropout layer and a 4-way softmax. The cell below rebuilds it from that diagram. **It was not part of the original runs:** the dropout rate and the training settings are not in the thesis, so the values here (dropout 0.3, 100 epochs, batch size 512) are assumptions.

In [ ]:
def build_bilstm(n_features, units=123, dropout=0.3):
    """BiLSTM rebuilt from the architecture diagram of the thesis (dropout rate assumed)."""
    model = keras.Sequential([
        keras.Input(shape=(n_features, 1)),
        layers.Bidirectional(layers.LSTM(units)),      # 2 x 123 = 246 outputs
        layers.Dropout(dropout),
        layers.Dense(len(EMOTIONS), activation="softmax"),
    ])
    model.compile(
        loss="categorical_crossentropy",
        optimizer="adam",
        metrics=["accuracy", keras.metrics.F1Score(average="macro", name="f1_score")],
    )
    return model


bilstm = build_bilstm(X_train.shape[1])
bilstm.summary()
history_bilstm = bilstm.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=100, batch_size=BATCH_SIZE)

**Results reported in the thesis** (test accuracy and F1 in %, same 1,025 test clips; produced by the original runs, which are not reproduced by the cell above):

| Features | LSTM accuracy | LSTM F1 | BiLSTM accuracy | BiLSTM F1 |
|---|---|---|---|---|
| MFCC | 61.73 | 56.41 | 59.22 | 52.02 |
| Chroma | 41.12 | 39.33 | 39.45 | 37.95 |
| Mel spectrogram | 61.03 | 59.65 | 59.67 | 46.47 |
| MFCC + Chroma | 61.95 | 58.02 | 58.73 | 50.30 |
| MFCC + Chroma + Mel | 56.98 | 50.53 | 57.17 | 49.75 |

The unidirectional LSTM is better than the BiLSTM in almost every case. One surviving run named `bilstmModel` (test accuracy 0.59, macro-F1 0.51) is very likely the BiLSTM on MFCC reported above (59.22 and 52.02). The architecture file saved next to it (`model.json`) holds the stacked LSTM of section 5, probably because a later run overwrote it.

> **No CNN.** No convolutional network appears in the thesis or in the recovered notebooks; the thesis lists CNNs on spectrograms as future work.